# Story River: Layout Algorithm & Matplotlib Preview

This notebook implements the Story River visualization algorithm:
1. (Optional) Extract knowledge from books if not already done
2. Load real KnowledgeBase from Supabase
3. Build chapter presence matrix
4. Compute Y ordering (characters that co-appear should be adjacent)
5. Compute lane paths with convergence (pull lanes together when sharing chapters)
6. Render with matplotlib
7. Export data structure for frontend SVG renderer

---

## STEP 0: Extract Knowledge (if needed)

**Run these cells first if your books don't have knowledge extracted yet.**

If all books are already extracted, skip to Step 1.

In [1]:
%load_ext autoreload
%autoreload 2


### Cell 0.1: Initialize LLM Client & Scan for EPUBs

In [2]:
from pathlib import Path
from src.ingestion.epub_parser import parse_epub
from src.knowledge.pipeline import extract_book_knowledge
from src.llm import create_llm_client
from src.config import settings
import asyncio

# Initialize LLM client
llm_client = create_llm_client()
print(f"LLM Provider: {settings.llm_provider}")
print(f"Extraction Model: {settings.extraction_model}")

# Scan for EPUBs
books_dir = Path("../uploads")  # or wherever you store EPUBs
epub_files = sorted(list(books_dir.glob("red-rising/*.epub")))
print(f"\nFound {len(epub_files)} EPUB files:")
for i, f in enumerate(epub_files):
    print(f"  {i}: {f.name}")

LLM Provider: gemini
Extraction Model: gemini-2.5-flash-lite

Found 3 EPUB files:
  0: book_0.epub
  1: book_1.epub
  2: book_2.epub


### Cell 0.2: Check Which Books Need Extraction

In [3]:
# Import needed for next steps
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')
from src.knowledge.store import load_knowledge

# Load current KB to check extraction status
# Change this to your series_id
series_id = "the-red-rising-saga"  # CHANGE THIS
kb = await load_knowledge(series_id)

def is_book_fully_extracted(kb, book_index, total_chapters):
    """Check if all chapters of a book are already extracted."""
    extracted = {ref.chapter_index for ref in kb.extracted_chapters if ref.book_index == book_index}
    return len(extracted) >= total_chapters

# Check each EPUB file
print(f"Series: {series_id}")
print(f"Current extraction status: {len(kb.extracted_chapters)} chapters extracted\n")

books_status = {}
for epub_path in epub_files:
    try:
        parsed = parse_epub(epub_path)
        # Assume 0-indexed order matching sorted filenames
        book_index = len(books_status)
        is_extracted = is_book_fully_extracted(kb, book_index, len(parsed))
        books_status[str(epub_path)] = {
            "book_index": book_index,
            "chapters": len(parsed),
            "extracted": is_extracted
        }
    except Exception as e:
        print(f"Error parsing {epub_path}: {e}")

print("Books in uploads directory:")
for epub_path, status in books_status.items():
    symbol = "✅" if status["extracted"] else "❌"
    print(f"{symbol} {Path(epub_path).name}")
    print(f"   Book index: {status['book_index']}, Chapters: {status['chapters']}, Extracted: {status['extracted']}")

Series: the-red-rising-saga
Current extraction status: 3 chapters extracted

Books in uploads directory:
❌ book_0.epub
   Book index: 0, Chapters: 50, Extracted: False
❌ book_1.epub
   Book index: 1, Chapters: 56, Extracted: False
❌ book_2.epub
   Book index: 2, Chapters: 82, Extracted: False


### Cell 0.3: Extract a Single Book

In [7]:
series_id

'the-red-rising-saga'

In [8]:
# MODIFY THESE:
book_to_extract_path = Path("../uploads/red-rising/book_0.epub")  # Change this path
book_index_to_use = 0  # Change this: which position in the series (0 = first book)

if book_to_extract_path.exists():
    print(f"Extracting knowledge from: {book_to_extract_path}")
    parsed_chapters = parse_epub(book_to_extract_path)
    print(f"  Parsed {len(parsed_chapters)} chapters")
    
    print(f"\nStarting extraction for book_index={book_index_to_use}...")
    result_kb = await extract_book_knowledge(
        chapters=parsed_chapters,
        canonical_series_id=series_id,
        book_index=book_index_to_use,
        client=llm_client,
        extraction_model=settings.extraction_model,
        concurrency=2,
    )
    
    kb = result_kb
    print(f"\n✅ Extraction complete!")
    print(f"  Characters: {len(kb.characters)}")
    print(f"  Relationships: {len(kb.relationships)}")
    print(f"  Extracted chapters: {len(kb.extracted_chapters)}")
else:
    print(f"❌ File not found: {book_to_extract_path}")
    print("Please update the path to match your EPUB location.")

Task exception was never retrieved
future: <Task finished name='Task-49' coro=<extract_book_knowledge.<locals>._safe_extract() done, defined at /Users/aravi/Documents/Workspace/Projects/book-lens/src/knowledge/pipeline.py:59> exception=RuntimeError('Gemini API returned empty content. Finish reason: FinishReason.MALFORMED_FUNCTION_CALL')>
Traceback (most recent call last):
  File "/Users/aravi/Documents/Workspace/Projects/book-lens/src/knowledge/pipeline.py", line 61, in _safe_extract
    return await extract_chapter(
           ^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/aravi/Documents/Workspace/Projects/book-lens/src/knowledge/extractor.py", line 326, in extract_chapter
    book_index,
                
  File "/Users/aravi/Documents/Workspace/Projects/book-lens/src/llm/gemini.py", line 92, in extract_structured
    kwargs["enum"] = schema["enum"]
RuntimeError: Gemini API returned empty content. Finish reason: FinishReason.MALFORMED_FUNCTION_CALL
Task exception was never retrieved
future: <

Extracting knowledge from: ../uploads/red-rising/book_0.epub
  Parsed 50 chapters

Starting extraction for book_index=0...


CancelledError: 

### Cell 0.4: Extract All Missing Books

In [ ]:
# Mapping of book file paths to their book_index in the series
# MODIFY THIS LIST:
books_to_extract = [
    (Path("../uploads/red-rising/book_1.epub"), 0),  # Red Rising (position 0)
    (Path("../uploads/red-rising/book_2.epub"), 1),  # Golden Son (position 1)
    # (Path("../uploads/red-rising/book_3.epub"), 2),  # Morning Star (position 2)
]

for epub_path, book_index in books_to_extract:
    if not epub_path.exists():
        print(f"⚠️  Skipping {epub_path} (not found)")
        continue
    
    parsed = parse_epub(epub_path)
    if is_book_fully_extracted(kb, book_index, len(parsed)):
        print(f"✅ {epub_path.name} already extracted (book_index={book_index})")
        continue
    
    print(f"\n🔄 Extracting {epub_path.name} (book_index={book_index})...")
    
    try:
        kb = await extract_book_knowledge(
            chapters=parsed,
            canonical_series_id=series_id,
            book_index=book_index,
            client=llm_client,
            extraction_model=settings.extraction_model,
            concurrency=2,
        )
        print(f"✅ {epub_path.name} extraction complete")
    except Exception as e:
        print(f"❌ {epub_path.name} extraction failed: {e}")

print(f"\n=== Final Knowledge Base State ===")
print(f"Characters: {len(kb.characters)}")
print(f"Relationships: {len(kb.relationships)}")
print(f"Extracted chapters: {len(kb.extracted_chapters)}")
print(f"\n✨ Ready to visualize with Story River!")

---

## STEP 1: Load the Knowledge Base

Run this cell after extraction is complete (or skip to here if already extracted).

In [ ]:
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

from src.knowledge.store import load_knowledge
import asyncio

# Load a test series (change series_id as needed)
series_id = "the-red-rising-saga"  # CHANGE THIS
kb = await load_knowledge(series_id)
print(f"Characters: {len(kb.characters)}")
print(f"Summaries: {len(kb.summaries)}")
print(f"Relationships: {len(kb.relationships)}")

## STEP 2: Build the Chapter Presence Matrix

In [ ]:
# chapter_key = (book_index, chapter_index)
# Source: kb.summaries[i].characters_present

presence = {}  # {char_name: set of (book, chapter) tuples}
all_chapters = []  # ordered list of (book_index, chapter_index)

for summary in sorted(kb.summaries, key=lambda s: (s.book_index, s.chapter_index)):
    ch_key = (summary.book_index, summary.chapter_index)
    if ch_key not in all_chapters:
        all_chapters.append(ch_key)
    for char_name in summary.characters_present:
        presence.setdefault(char_name, set()).add(ch_key)

print(f"Total chapters: {len(all_chapters)}")
print(f"Characters with presence data: {len(presence)}")
print(f"Sample chapters: {all_chapters[:5]}")

## STEP 3: Compute Y Ordering (Greedy Co-occurrence)

In [ ]:
from collections import defaultdict

# Build co-occurrence matrix: how many chapters do char_a and char_b share?
co = defaultdict(lambda: defaultdict(int))
for summary in kb.summaries:
    chars = summary.characters_present
    for i, a in enumerate(chars):
        for b in chars[i+1:]:
            co[a][b] += 1
            co[b][a] += 1

def greedy_y_order(characters, co):
    """
    Greedy ordering: start with the character with most total co-occurrences (the protagonist),
    then always add the character most connected to the already-placed set.
    """
    if not characters:
        return []
    
    # Seed: character with most co-occurrences overall
    seed = max(characters, key=lambda c: sum(co[c].values()))
    ordered = [seed]
    remaining = set(characters) - {seed}
    
    while remaining:
        # Score each remaining character by total co-occurrence with already-placed chars
        best = max(remaining, key=lambda c: sum(co[c].get(p, 0) for p in ordered))
        ordered.append(best)
        remaining.remove(best)
    
    return ordered  # index = Y position

char_order = greedy_y_order(list(presence.keys()), co)
y_pos = {char: i for i, char in enumerate(char_order)}

print(f"Y ordering (first 10): {char_order[:10]}")

## STEP 4: Compute Lane Paths with Convergence

In [ ]:
def compute_lane_segments(char_order, presence, all_chapters):
    """
    Returns: {char_name: [(x, y_base, y_actual, is_present)]}
    x = chapter index along X axis
    y_base = resting Y position
    y_actual = Y during convergence (pulled toward group midpoint when sharing a chapter)
    is_present = True if character appears in this chapter (draw solid lane)
    """
    segments = {char: [] for char in char_order}
    pull_strength = 0.35  # how far lanes bend toward each other (0=none, 1=fully merge)
    
    for x, ch_key in enumerate(all_chapters):
        # Who is present in this chapter?
        present_here = [c for c in char_order if ch_key in presence.get(c, set())]
        
        # Compute convergence midpoint if multiple characters present
        if len(present_here) > 1:
            midpoint_y = sum(y_pos[c] for c in present_here) / len(present_here)
        else:
            midpoint_y = None
        
        for char in char_order:
            base_y = y_pos[char]
            if ch_key in presence.get(char, set()):
                if len(present_here) > 1 and midpoint_y is not None:
                    actual_y = base_y + (midpoint_y - base_y) * pull_strength
                else:
                    actual_y = base_y
                segments[char].append((x, base_y, actual_y, True))
            else:
                segments[char].append((x, base_y, base_y, False))
    
    return segments

segments = compute_lane_segments(char_order, presence, all_chapters)
print(f"Computed segments for {len(segments)} characters")
print(f"Sample segment (first char, first 5 chapters): {segments[char_order[0]][:5]}")

## STEP 5: Render with Matplotlib

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.path import Path
import matplotlib.patheffects as pe
import numpy as np

# Create figure with appropriate size
fig, ax = plt.subplots(figsize=(24, max(6, len(char_order) * 0.6)))
ax.set_facecolor('#080b14')
fig.set_facecolor('#080b14')

# Faction colors (hash-based for determinism)
faction_colors = {}
palette = ['#c9a84c', '#4c8bc9', '#c94c4c', '#4cc9a8', '#c94c8b', '#8b4cc9']
for char in kb.characters:
    if char.faction:
        faction_colors[char.name] = palette[hash(char.faction) % len(palette)]
    else:
        faction_colors[char.name] = '#7a7a8a'

# Draw lanes
for char, seg in segments.items():
    xs = [s[0] for s in seg]
    ys = [s[2] for s in seg]  # y_actual
    color = faction_colors.get(char, '#7a7a8a')
    
    # Draw the lane (skip gaps where character is absent)
    for i in range(len(seg) - 1):
        if seg[i][3]:  # is_present
            ax.plot([xs[i], xs[i+1]], [ys[i], ys[i+1]], 
                    color=color, linewidth=2, alpha=0.8,
                    path_effects=[pe.withStroke(linewidth=4, foreground=color, alpha=0.2)])

# Add key event markers (first appearance)
for char in kb.characters:
    if char.first_appearance:
        ch_key = (char.first_appearance.book_index, char.first_appearance.chapter_index)
        if ch_key in all_chapters:
            x = all_chapters.index(ch_key)
            y = y_pos.get(char.name)
            if y is not None:
                ax.scatter(x, y, s=80, color='white', zorder=5, alpha=0.9,
                          edgecolors=faction_colors.get(char.name, '#7a7a8a'), linewidths=1.5)

# Add character labels on the left
for char in char_order:
    y = y_pos[char]
    color = faction_colors.get(char, '#7a7a8a')
    ax.text(-1, y, char, color=color, fontsize=7, ha='right', va='center')

ax.set_xlim(-5, len(all_chapters))
ax.set_ylim(-1, len(char_order))
ax.set_xlabel('Chapter', color='#e8e4db', fontsize=10)
ax.set_ylabel('Character', color='#e8e4db', fontsize=10)
ax.tick_params(colors='#e8e4db')

plt.tight_layout()
plt.savefig('story_river_preview.png', dpi=150, bbox_inches='tight', facecolor='#080b14')
plt.show()

print("Story River preview saved to story_river_preview.png")

## STEP 6: Export Data Structure for Frontend

In [ ]:
def export_river_data(segments, all_chapters, kb, y_pos, faction_colors):
    """Build the JSON structure the SVG renderer in the browser will use."""
    lanes = []
    for char_name, seg in segments.items():
        lanes.append({
            "character": char_name,
            "y_base": y_pos[char_name],
            "color": faction_colors.get(char_name, '#7a7a8a'),
            "segments": [
                {
                    "x": x,
                    "y": y_actual,
                    "present": present
                }
                for x, y_base, y_actual, present in seg
            ]
        })
    
    markers = []
    for char in kb.characters:
        if char.first_appearance:
            ch_key = (char.first_appearance.book_index, char.first_appearance.chapter_index)
            if ch_key in all_chapters:
                markers.append({
                    "x": all_chapters.index(ch_key),
                    "y": y_pos.get(char.name),
                    "type": "first_appearance",
                    "label": f"{char.name} first appears",
                    "character": char.name
                })
    
    chapter_labels = [
        {"x": i, "book": ch[0], "chapter": ch[1]}
        for i, ch in enumerate(all_chapters)
    ]
    
    return {
        "lanes": lanes,
        "markers": markers,
        "chapters": chapter_labels
    }

river_data = export_river_data(segments, all_chapters, kb, y_pos, faction_colors)

# Print preview
import json
print("River data structure (preview):")
print(json.dumps(river_data, indent=2)[:2000])  # First 2000 chars
print(f"\nTotal data size: {len(json.dumps(river_data))} bytes")

## Summary

✅ Story River layout complete. The notebook demonstrates:

- **Y ordering**: Characters that appear together are placed vertically adjacent (minimizes line crossings)
- **Convergence**: Lanes pull together when characters share a chapter (uses `pull_strength = 0.35`)
- **Matplotlib preview**: Visual validation of the layout
- **Frontend-ready JSON**: `river_data` structure is ready for SVG rendering

### Workflow:
1. **Run STEP 0 cells** (0.1→0.2→0.3/0.4) to extract knowledge from books
2. **Run STEP 1** to load the knowledge base
3. **Run STEPS 2-6** to build and visualize the Story River

### Customization:
- Adjust `pull_strength` (Step 4) if convergence curves need to be more/less pronounced
- Tweak `palette` colors (Step 5) to match your theme
- Change `series_id` in Steps 0.2 and 1 to visualize different series

### Next steps:
1. Review the matplotlib preview — does the Y ordering look right?
2. Once validated, move on to Notebook B (Character Constellation)